# 05_docs_split

对应 `index.md` 第 5 阶段：文档加载、文本切分。

笔记本在 `codes/langchain/05_docs_split/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

本阶段不调用 ChatModel。核心对象：`Document`、`RecursiveCharacterTextSplitter` / `CharacterTextSplitter`。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "05_docs_split" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "langchain" / "05_docs_split"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT


PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/langchain/05_docs_split')

## 1. 构造 Document

用 `langchain_core.documents.Document` 建一个文档：`page_content` 为 `LangChain 用 Document 表示一段文本。`，`metadata` 为 `{"source": "memo", "page": 1}`。

打印 `doc.page_content` 和 `doc.metadata`。


In [2]:
from langchain_core.documents import Document

# 作答

doc = Document(
    page_content="LangChain 用 Document 表示一段文本。",
    metadata={"source": "memo", "page": 1},
)

print(doc.page_content)
print(doc.metadata)
#评阅
# 对。page_content 与 metadata 都建对了。

#参考答案
# doc = Document(
#     page_content="LangChain 用 Document 表示一段文本。",
#     metadata={"source": "memo", "page": 1},
# )
# print(doc.page_content)
# print(doc.metadata)


LangChain 用 Document 表示一段文本。
{'source': 'memo', 'page': 1}


## 2. 从文件读成 Document

前置代码已把一段说明写到 `ROOT / "q2" / "note.txt"`。

读出该文件全文，建成一个 `Document`：`page_content` 为文件内容，`metadata` 里至少含 `"source"`，值为该文件的字符串路径（可用 `str(path)`）。

打印 `doc.page_content` 与 `doc.metadata["source"]`。


In [4]:
from pathlib import Path

from langchain_core.documents import Document

q2_dir = ROOT / "q2"
q2_dir.mkdir(parents=True, exist_ok=True)
note_path = q2_dir / "note.txt"
note_path.write_text(
    "RAG 先把材料切成小块，再检索相关块喂给模型。\n",
    encoding="utf-8",
)

# 作答

doc = Document(
    note_path.read_text(encoding="utf-8"),
    metadata={"source": str(note_path)},
)

print(doc.page_content)
print(doc.metadata["source"])
#评阅
# 对。读文件进 page_content，source 用路径字符串。

#参考答案
# doc = Document(
#     page_content=note_path.read_text(encoding="utf-8"),
#     metadata={"source": str(note_path)},
# )
# print(doc.page_content)
# print(doc.metadata["source"])


RAG 先把材料切成小块，再检索相关块喂给模型。

/Users/keyficller/Documents/AEFS-Notes/codes/langchain/05_docs_split/q2/note.txt


## 3. RecursiveCharacterTextSplitter 切纯文本

已有长字符串 `text`。用 `RecursiveCharacterTextSplitter`，`chunk_size=40`，`chunk_overlap=10`，对 `text` 调用 `split_text`。

打印切出的块数，以及每一块的长度（`len(chunk)`）与内容。


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = (
    "第一段讲什么是 Document。"
    "第二段讲为什么要切分长文。"
    "第三段讲 chunk_size 与 overlap 如何影响块数。"
    "第四段提醒下一阶段才会接上向量库。"
)

# 作答

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 40,
    chunk_overlap = 10
)

for chunk in splitter.split_text(text):
    print(chunk)
#评阅
# 缺一半。split_text 与参数对了，但题面还要打印块数，以及每块的 len(chunk)。

#参考答案
# splitter = RecursiveCharacterTextSplitter(chunk_size=40, chunk_overlap=10)
# chunks = splitter.split_text(text)
# print(len(chunks))
# for chunk in chunks:
#     print(len(chunk), chunk)


第一段讲什么是 Document。第二段讲为什么要切分长文。第三段讲
chunk_size 与 overlap
与 overlap 如何影响块数。第四段提醒下一阶段才会接上向量库。


## 4. split_documents 保留 metadata

已有文档列表 `docs`。用 `RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=4)` 调用 `split_documents(docs)`。

对每个切出的块，打印 `page_content` 和 `metadata`（应仍带上原来的 `source`）。


In [7]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = [
    Document(
        page_content=(
            "苹果适合做派。香蕉适合做奶昔。橙子适合榨汁。葡萄适合生吃。"
        ),
        metadata={"source": "fruits.txt"},
    )
]

# 作答

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 20,
    chunk_overlap = 4
)

for doc in splitter.split_documents(docs):
    print(doc.page_content)
    print(doc.metadata)
#评阅
# 对。split_documents 后 metadata 里仍有 source。

#参考答案
# splitter = RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=4)
# for doc in splitter.split_documents(docs):
#     print(doc.page_content)
#     print(doc.metadata)


苹果适合做派。香蕉适合做奶昔。橙子适合榨
{'source': 'fruits.txt'}
子适合榨汁。葡萄适合生吃。
{'source': 'fruits.txt'}


## 5. CharacterTextSplitter 按分隔符切

已有带空行分段的 `text`。用 `CharacterTextSplitter`，`separator="\n\n"`，`chunk_size=40`，`chunk_overlap=0`，对 `text` 调用 `split_text`。

打印每一块（应大致按空行分成三段）。


In [8]:
from langchain_text_splitters import CharacterTextSplitter

text = (
    "段落甲：Document 有 page_content 和 metadata。\n\n"
    "段落乙：切分器把长文变成可检索的小块。\n\n"
    "段落丙：Recursive 与 Character 的切法不同。"
)

# 作答

splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size = 40,
    chunk_overlap = 0
)

for chunk in splitter.split_text(text):
    print(chunk)
#评阅
# 对。按 \n\n 切开，参数与 split_text 都对。

#参考答案
# splitter = CharacterTextSplitter(
#     separator="\n\n", chunk_size=40, chunk_overlap=0
# )
# for chunk in splitter.split_text(text):
#     print(chunk)


段落甲：Document 有 page_content 和 metadata。
段落乙：切分器把长文变成可检索的小块。
段落丙：Recursive 与 Character 的切法不同。


## 6. create_documents

已有字符串列表 `texts` 与对应 `metadatas`。用 `RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)` 的 `create_documents(texts, metadatas=metadatas)` 生成文档列表。

依次打印每个文档的 `page_content` 与 `metadata`。


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

texts = [
    "短文一：关于加载。",
    "短文二：关于切分。",
]
metadatas = [
    {"source": "a.txt", "idx": 0},
    {"source": "b.txt", "idx": 1},
]

# 作答
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 0
)

docs = splitter.create_documents(texts, metadatas=metadatas)

for doc in docs:
    print(doc.page_content)
    print(doc.metadata)
#评阅
# 对。create_documents 带上了对应 metadatas。

#参考答案
# splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
# docs = splitter.create_documents(texts, metadatas=metadatas)
# for doc in docs:
#     print(doc.page_content)
#     print(doc.metadata)


短文一：关于加载。
{'source': 'a.txt', 'idx': 0}
短文二：关于切分。
{'source': 'b.txt', 'idx': 1}
